In [ ]:
!pip install -q ragas langchain langchain-community llama-parse llama-index-core sentence-transformers chromadb pandas datasets

In [ ]:
!pip install langchain-groq

In [ ]:
import os
import re
import torch
from llama_parse import LlamaParse
from langchain_core.documents import Document as LangchainDocument
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Parsing

In [ ]:
os.environ["LLAMA_CLOUD_API_KEY"] = "YOUR_LLAMA_CLOUD_KEY"

def clean_text_content(text):
    if not text:
        return ""

    text = re.sub(r'<sup[^>]*>.*?</sup>', '', text, flags=re.IGNORECASE)
    text = re.sub(r'<br\s*/?>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'</?(b|i|u|strong|em)>', '', text, flags=re.IGNORECASE)

    lines = text.split('\n')
    cleaned_lines = []

    for line in lines:
        stripped_line = line.strip()

        if stripped_line.isdigit() and len(stripped_line) <= 3:
            continue

        if "HEARTS – D" in stripped_line or "HEARTS - D" in stripped_line:
            continue

        if "DEFINITION AND DIAGNOSIS OF DIABETES MELLITUS AND INTERMEDIATE HYPERGLYCEMIA" in stripped_line.upper():
            continue

        cleaned_lines.append(line)

    return '\n'.join(cleaned_lines).strip()

def parse_pdf_with_metadata(file_path):
    parser = LlamaParse(
        result_type="markdown",
        verbose=True,
        language="en",
    )

    documents = parser.load_data(file_path)

    for i, doc in enumerate(documents):
        doc.metadata["file_name"] = os.path.basename(file_path)
        doc.metadata["page"] = i + 1

        cleaned_text = clean_text_content(doc.text)
        doc.set_content(cleaned_text)

    return documents

pdf_files = [
    "/kaggle/input/datasets/ahmedh72/ai-hackathon/General classification.pdf",
    "/kaggle/input/datasets/ahmedh72/ai-hackathon/Type 2_20.1.pdf",
    "/kaggle/input/datasets/ahmedh72/ai-hackathon/Type 2_3.1.pdf"
]

all_parsed_documents = []

for pdf_file in pdf_files:
    if os.path.exists(pdf_file):
        print(f"\n--- جاري معالجة الملف: {pdf_file} ---")
        docs = parse_pdf_with_metadata(pdf_file)
        all_parsed_documents.extend(docs)
    else:
        print(f"\n[تنبيه] الملف غير موجود: {pdf_file}")

print(f"\nإجمالي عدد الصفحات المستخرجة والممنظفة من كل الملفات: {len(all_parsed_documents)}")

In [ ]:
target_page = 1
docs = all_parsed_documents[target_page]

print(f"--- الميتا داتا ---")
print(f"اسم الملف: {docs.metadata['file_name']}")
print(f"رقم الصفحة: {docs.metadata['page']}")
print(f"--- النص/الجدول (Markdown) ---")
print(docs.text)

# Fixed Chunking

In [ ]:
langchain_docs = [
    LangchainDocument(
        page_content=doc.text,
        metadata=doc.metadata
    )
    for doc in all_parsed_documents
]

text_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(langchain_docs)
print(f"إجمالي عدد الـ Chunks الناتجة: {len(chunks)}")
print(chunks[80].page_content)
print(chunks[80].metadata)

# Recursive Chunking

In [ ]:
langchain_docs = [
    LangchainDocument(
        page_content=doc.text,
        metadata=doc.metadata
    )
    for doc in all_parsed_documents
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(langchain_docs)

print(f"إجمالي عدد الـ Chunks الناتجة: {len(chunks)}")

if len(chunks) > 80:
    print("\n--- عينة من Chunk رقم 80 ---")
    print(chunks[80].page_content)
    print("\n--- الميتا داتا الخاصة به ---")
    print(chunks[80].metadata)
else:
    print(f"عدد الـ Chunks الكلي ({len(chunks)}) أقل من 80، جرب تطبّق على رقم أصغر مثل chunks[0]")

# Embedding Models

**BAAI**

In [ ]:
model = SentenceTransformer("BAAI/bge-m3")

texts = [chunk.page_content if hasattr(chunk, 'page_content') else chunk for chunk in chunks]

embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(embeddings.shape)

**multilingual**

In [ ]:
model = SentenceTransformer("intfloat/multilingual-e5-large")

texts = [chunk.page_content if hasattr(chunk, 'page_content') else chunk for chunk in chunks]

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cuda"}, 
    encode_kwargs={"normalize_embeddings": True}
)

In [ ]:
model = SentenceTransformer(
     "Qwen/Qwen3-Embedding-4B",
     device="cuda",
     trust_remote_code=True
)

texts = [chunk.page_content if hasattr(chunk, 'page_content') else chunk for chunk in chunks]

embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(embeddings.shape)

# Vector DB

In [ ]:
persist_directory = './rag_vector_db'

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

#vectorstore.persist()

print(f"The ChromaDB vector database was successfully created and stored at: {persist_directory}")

# Retrieval

In [ ]:
import json

test_dataset = [
    {
        "id": 1,
        "query": "What are the current World Health Organization (WHO) diagnostic criteria for diabetes regarding fasting plasma glucose and 2-hour plasma glucose?",
        "ground_truth": "According to the provided sources, the diagnostic criteria include a fasting plasma glucose value of $\\ge 7.0$ mmol/l ($126$ mg/dl) and/or a 2-hour plasma glucose value of $\\ge 11.1$ mmol/l ($200$ mg/dl) after a 75g oral glucose tolerance test[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "1"
    },
    {
        "id": 2,
        "query": "Why is the oral glucose tolerance test (OGTT) retained as a diagnostic test, and what percentage of previously undiagnosed diabetes cases does fasting plasma glucose alone fail to diagnose?",
        "ground_truth": "The OGTT is retained because fasting plasma glucose alone fails to diagnose up to 30% to 50% of people with diabetes who have elevated 2-hour post-load glucose levels[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "1"
    },
    {
        "id": 3,
        "query": "According to the WHO technical report, what were the main reasons why the fasting plasma glucose cut-point for Impaired Fasting Glucose (IFG) was recommended to remain at 6.1 mmol/l rather than being lowered to 5.6 mmol/l?",
        "ground_truth": "The cut-point was kept at 6.1 mmol/l to maintain consistency with existing epidemiological data, prevent a massive surge in the diagnosed population, and because lowering it to 5.6 mmol/l did not provide a significantly better risk prediction threshold across diverse populations[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "1"
    },
    {
        "id": 4,
        "query": "How do venous and capillary plasma glucose measurements differ in the fasting versus the non-fasting state?",
        "ground_truth": "In the fasting state, venous and capillary whole blood and plasma glucose values are very close or virtually identical. In the non-fasting (postprandial) state, capillary glucose values are higher than venous values due to tissue glucose uptake[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "1"
    },
    {
        "id": 5,
        "query": "Why is HbA1c currently not considered a suitable diagnostic test for diabetes or intermediate hyperglycaemia by the WHO group?",
        "ground_truth": "HbA1c was not recommended due to a lack of international standardization of assays, high costs in many countries, and insufficient consensus on standardized cut-off points globally at the time of the report[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "1"
    },
    {
        "id": 6,
        "query": "What are the three potential approaches to screening for type 2 diabetes outlined in the WHO and IDF document?",
        "ground_truth": "The three approaches are: opportunistic screening of individuals attending health care settings, systematic screening of high-risk individuals, and population-based mass screening[cite: 8].",
        "source": "Screening for Type 2 Diabetes (WHO/IDF)",
        "page": "2"
    },
    {
        "id": 7,
        "query": "According to evaluation data, what are the general statistical performance limitations of using urinary glucose as a screening test for undiagnosed diabetes?",
        "ground_truth": "Urinary glucose has poor sensitivity, meaning it misses a large number of true cases, and low positive predictive value because glycosuria only occurs once renal thresholds are exceeded, making it an unreliable screening tool[cite: 8].",
        "source": "Screening for Type 2 Diabetes (WHO/IDF)",
        "page": "2"
    },
    {
        "id": 8,
        "query": "What significant differences exist between the 1999 WHO criteria and the 2003 American Diabetes Association (ADA) criteria concerning the definition of Impaired Fasting Glucose (IFG)?",
        "ground_truth": "The 1999 WHO criteria defined IFG as a fasting plasma glucose of 6.1 to 6.9 mmol/l, whereas the 2003 ADA criteria lowered the lower bound of IFG to 5.6 mmol/l (5.6 to 6.9 mmol/l)[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "1"
    },
    {
        "id": 9,
        "query": "What did the DECODE study report regarding the relationship between mortality and glucose levels, and what fasting plasma glucose range showed the lowest death rates?",
        "ground_truth": "The DECODE study showed that mortality risk increases with both elevated fasting and 2-hour post-load glucose levels, with the lowest death rates generally observed within normal fasting glucose ranges below 6.0 mmol/l[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "1"
    },
    {
        "id": 10,
        "query": "How do lifestyle interventions compare to medications in preventing or delaying the transition from Impaired Glucose Tolerance (IGT) to diabetes according to clinical trials?",
        "ground_truth": "Clinical trials demonstrated that intensive lifestyle interventions (diet and physical activity) are generally more effective or equally effective at significantly reducing the incidence of progression to type 2 diabetes compared to pharmacological agents alone[cite: 8].",
        "source": "Screening for Type 2 Diabetes (WHO/IDF)",
        "page": "2"
    },
    {
        "id": 11,
        "query": "What are the core clinical objectives of implementing structured population screening programs for gestational diabetes mellitus?",
        "ground_truth": "Core objectives include identifying unrecognized carbohydrate intolerance during pregnancy, preventing maternal-fetal complications, and reducing perinatal morbidity and mortality[cite: 7].",
        "source": "Diagnostic Criteria and Classification of Hyperglycaemia First Detected in Pregnancy (WHO Guidelines)",
        "page": "14"
    },
    {
        "id": 12,
        "query": "Why are fasting plasma glucose thresholds alone considered insufficient for diagnosing gestational diabetes during early prenatal visits?",
        "ground_truth": "Fasting plasma glucose alone can miss a significant proportion of gestational diabetes cases where postprandial hyperglycemia is the primary abnormality, making an OGTT necessary[cite: 7].",
        "source": "Diagnostic Criteria and Classification of Hyperglycaemia First Detected in Pregnancy (WHO Guidelines)",
        "page": "15"
    },
    {
        "id": 13,
        "query": "What specific diagnostic cut-offs were proposed by the International Association of Diabetes and Pregnancy Study Groups (IADPSG) for a 75g OGTT at 24-28 weeks gestation?",
        "ground_truth": "The cut-offs are: fasting $\\ge 5.1$ mmol/l, 1-hour $\\ge 10.0$ mmol/l, and 2-hour $\\ge 8.5$ mmol/l[cite: 7].",
        "source": "Diagnostic Criteria and Classification of Hyperglycaemia First Detected in Pregnancy (WHO Guidelines)",
        "page": "16"
    },
    {
        "id": 14,
        "query": "How does the classification distinguish between diabetes mellitus diagnosed for the first time in pregnancy versus gestational diabetes mellitus (GDM)?",
        "ground_truth": "Diabetes mellitus diagnosed in pregnancy meets standard WHO non-pregnant diagnostic criteria (e.g., fasting $\\ge 7.0$ mmol/l), whereas GDM represents lesser degrees of hyperglycemia first recognized during pregnancy[cite: 7].",
        "source": "Diagnostic Criteria and Classification of Hyperglycaemia First Detected in Pregnancy (WHO Guidelines)",
        "page": "18"
    },
    {
        "id": 15,
        "query": "What are the primary logistical and economic barriers associated with implementing universal 75g OGTT screening for all pregnant women in resource-limited settings?",
        "ground_truth": "Barriers include the requirement for fasting states, multiple blood draws, laboratory processing capacity, high financial costs, and patient compliance challenges[cite: 7].",
        "source": "Diagnostic Criteria and Classification of Hyperglycaemia First Detected in Pregnancy (WHO Guidelines)",
        "page": "21"
    },
    {
        "id": 16,
        "query": "What epidemiological trends were highlighted regarding the global prevalence of type 2 diabetes and its shifting age demographics over recent decades?",
        "ground_truth": "The prevalence of type 2 diabetes has increased dramatically worldwide, affecting younger populations and expanding rapidly in low- and middle-income countries due to urbanization and lifestyle changes[cite: 8].",
        "source": "Screening for Type 2 Diabetes (WHO/IDF)",
        "page": "5"
    },
    {
        "id": 17,
        "query": "What are the main risk factors recommended for consideration when designing targeted or opportunistic screening strategies for high-risk groups?",
        "ground_truth": "Key risk factors include age (e.g., over 45), family history of diabetes, history of gestational diabetes, membership in high-risk ethnic groups, obesity, and physical inactivity[cite: 8].",
        "source": "Screening for Type 2 Diabetes (WHO/IDF)",
        "page": "7"
    },
    {
        "id": 18,
        "query": "How do cost-effectiveness analyses evaluate population-based mass screening versus targeted screening for type 2 diabetes?",
        "ground_truth": "Cost-effectiveness analyses generally indicate that population-based mass screening is inefficient and expensive, whereas targeted screening of high-risk individuals offers a much better use of healthcare resources[cite: 8].",
        "source": "Screening for Type 2 Diabetes (WHO/IDF)",
        "page": "11"
    },
    {
        "id": 19,
        "query": "What role do risk assessment scoring questionnaires play prior to conducting biochemical blood tests in screening protocols?",
        "ground_truth": "Non-invasive risk scoring tools help stratify individuals based on risk factors, reducing the number of unnecessary blood tests and optimizing resource allocation for definitive laboratory testing[cite: 8].",
        "source": "Screening for Type 2 Diabetes (WHO/IDF)",
        "page": "9"
    },
    {
        "id": 20,
        "query": "What considerations are outlined regarding the psychological and social impacts of labeling an individual with pre-diabetes or diabetes through screening?",
        "ground_truth": "Screening can induce anxiety, stigmatization, and potential impacts on insurance or employment status, which must be balanced against the clinical benefits of early detection[cite: 8].",
        "source": "Screening for Type 2 Diabetes (WHO/IDF)",
        "page": "14"
    },
    {
        "id": 21,
        "query": "What biochemical classifications are used to categorize fasting plasma glucose values that fall below the diagnostic threshold for diabetes but above normal limits?",
        "ground_truth": "Values are categorized as Impaired Fasting Glucose (IFG) when they fall between specific intermediate boundaries, indicating dysglycemia[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "3"
    },
    {
        "id": 22,
        "query": "How do random blood glucose measurements combined with classic symptoms of hyperglycemia contribute to the immediate clinical diagnosis of diabetes?",
        "ground_truth": "A random plasma glucose $\\ge 11.1$ mmol/l in the presence of classic symptoms (polyuria, polydipsia, unexplained weight loss) is sufficient to establish a diagnosis of diabetes without requiring an OGTT[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "4"
    },
    {
        "id": 23,
        "query": "What standardization issues regarding glucose oxidase and hexokinase laboratory assay methods were discussed in the technical report?",
        "ground_truth": "While enzymatic methods like hexokinase are highly accurate, variations in sample handling, hemolysis, and delayed separation of plasma from red blood cells can introduce analytical errors if not strictly controlled[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "8"
    },
    {
        "id": 24,
        "query": "What guidance does the report provide concerning the necessity of repeating diagnostic tests to confirm diabetes in asymptomatic individuals?",
        "ground_truth": "In the absence of unequivocal hyperglycemia with acute metabolic decompensation, diagnosis should be confirmed on a subsequent day by repeating the diagnostic test to rule out transient laboratory or biological variations[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "5"
    },
    {
        "id": 25,
        "query": "How are secondary types of diabetes associated with exocrine pancreatic disease or endocrinopathies distinguished in the etiological classification scheme?",
        "ground_truth": "They are classified under specific secondary categories separate from type 1 and type 2, based on the identifiable underlying pathological process or specific drug/chemical induction[cite: 7].",
        "source": "Definition, Diagnosis and Classification of Diabetes Mellitus and its Complications (WHO Technical Report Series)",
        "page": "12"
    },
    {
        "id": 26,
        "query": "What recommendations are made regarding follow-up intervals for individuals identified with Impaired Glucose Tolerance (IGT) during screening programs?",
        "ground_truth": "Individuals with IGT should receive lifestyle counseling and be re-evaluated with periodic glucose tolerance testing annually due to their high annual conversion rate to overt type 2 diabetes[cite: 8].",
        "source": "Screening for Type 2 Diabetes (WHO/IDF)",
        "page": "18"
    },
    {
        "id": 27,
        "query": "What evidence supports the integration of community-based health workers in executing decentralized screening initiatives?",
        "ground_truth": "Community health workers effectively improve outreach, increase screening uptake in underserved rural or marginalized populations, and facilitate timely referral pathways[cite: 8].",
        "source": "Screening for Type 2 Diabetes (WHO/IDF)",
        "page": "22"
    },
    {
        "id": 28,
        "query": "What analytical performance criteria are specified for point-of-care (POC) blood glucose meters when utilized in formal screening contexts?",
        "ground_truth": "POC devices must meet strict quality control standards with minimal acceptable coefficient of variation and correlation with venous laboratory reference assays to avoid misclassification[cite: 8].",
        "source": "Screening for Type 2 Diabetes (WHO/IDF)",
        "page": "25"
    },
    {
        "id": 29,
        "query": "What specific adverse neonatal outcomes are strongly correlated with maternal hyperglycemia levels below those diagnostic of overt diabetes during pregnancy?",
        "ground_truth": "Outcomes include macrosomia, primary cesarean delivery, neonatal hypoglycemia, hyperbilirubinemia, and increased cord serum C-peptide levels[cite: 7].",
        "source": "Diagnostic Criteria and Classification of Hyperglycaemia First Detected in Pregnancy (WHO Guidelines)",
        "page": "32"
    },
    {
        "id": 30,
        "query": "How do post-partum follow-up testing recommendations differ for women who experienced gestational diabetes compared to normoglycemic pregnancies?",
        "ground_truth": "Women with prior GDM require a 75g OGTT at 6 to 12 weeks post-partum and lifelong periodic monitoring because they carry a substantially elevated lifetime risk of developing type 2 diabetes[cite: 7].",
        "source": "Diagnostic Criteria and Classification of Hyperglycaemia First Detected in Pregnancy (WHO Guidelines)",
        "page": "45"
    }
]

# حفظ مجموعة الاختبار
with open("rag_test_set.json", "w", encoding="utf-8") as f:
    json.dump(test_dataset, f, ensure_ascii=False, indent=4)

print("Saved 30 test cases successfully.")

In [ ]:
import json
import os
import time
import pandas as pd
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_recall, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI

# ضع مفتاح Google API الخاص بك هنا أو اجعله متغير بيئة
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"  # ضع مفتاحك هنا

# ============================================================
# 0. تأكد إن الـ API key موجود
# ============================================================
if not os.environ.get("GOOGLE_API_KEY"):
    raise EnvironmentError("GOOGLE_API_KEY is missing!")

# ============================================================
# 1. تحميل مجموعة الاختبار
# ============================================================
with open("rag_test_set.json", "r", encoding="utf-8") as f:
    test_dataset_json = json.load(f)

eval_questions = [item["query"] for item in test_dataset_json]
eval_ground_truths = [item["ground_truth"] for item in test_dataset_json]

sample_text = " ".join([item["ground_truth"] for item in test_dataset_json])

# ============================================================
# 2. تجهيز LLM (Gemini) و Embeddings لـ ragas
# ============================================================
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0,
    google_api_key=os.environ["GOOGLE_API_KEY"],
)

ragas_llm = LangchainLLMWrapper(gemini_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# ============================================================
# 3. الإعدادات المطلوب اختبارها
# ============================================================
configs = [
    {"chunk_size": 700, "overlap": 100}
]

results_summary = []

# ============================================================
# 4. حلقة التكرار على الإعدادات والاستراتيجيات
# ============================================================
for cfg in configs:
    c_size = cfg["chunk_size"]
    c_overlap = cfg["overlap"]

    strategies = {
        "Fixed Chunking": CharacterTextSplitter(
            chunk_size=c_size, chunk_overlap=c_overlap, separator=""
        ),
        "Recursive Chunking": RecursiveCharacterTextSplitter(
            chunk_size=c_size, chunk_overlap=c_overlap
        ),
    }

    for strat_name, splitter in strategies.items():
        run_id = f"{strat_name}_{c_size}_{c_overlap}".replace(" ", "_")
        print(f"\n=== Running: {run_id} ===")
        time.sleep(2)  

        chunks = splitter.create_documents([sample_text])

        vectorstore = Chroma.from_documents(
            chunks, embeddings, collection_name=run_id
        )
        retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

        retrieved_contexts = []
        for q in eval_questions:
            docs = retriever.invoke(q)
            retrieved_contexts.append([doc.page_content for doc in docs])

        eval_dataset = Dataset.from_dict(
            {
                "question": eval_questions,
                "answer": eval_ground_truths,
                "contexts": retrieved_contexts,
                "ground_truth": eval_ground_truths,
            }
        )

        try:
            evaluation_result = evaluate(
                dataset=eval_dataset,
                metrics=[context_recall, context_precision],
                llm=ragas_llm,
                embeddings=ragas_embeddings,
            )

            df_res = evaluation_result.to_pandas()
            mean_recall = df_res["context_recall"].mean()
            mean_precision = df_res["context_precision"].mean()

        except Exception as e:
            print(f"[{run_id}] FAILED: {e}")
            mean_recall = 0.0
            mean_precision = 0.0

        finally:
            try:
                vectorstore.delete_collection()
            except Exception:
                pass

        results_summary.append(
            {
                "Chunking Strategy": strat_name,
                "Chunk Size": c_size,
                "Overlap": c_overlap,
                "Context Recall": round(mean_recall, 4),
                "Context Precision": round(mean_precision, 4),
            }
        )

# ============================================================
# 5. عرض النتائج
# ============================================================
results_df = pd.DataFrame(results_summary)
markdown_table = results_df.to_markdown(index=False)

print("\n### جدول مقارنة أداء الـ Chunking باستخدام Ragas و Gemini\n")
print(markdown_table)

results_df.to_csv("chunking_eval_results_gemini.csv", index=False)
print("\nتم حفظ النتائج في chunking_eval_results_gemini.csv")

/tmp/ipykernel_238/176091344.py:10: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall, context_precision
/tmp/ipykernel_238/176091344.py:10: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_recall, context_precision


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_238/176091344.py:46: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(gemini_llm)
/tmp/ipykernel_238/176091344.py:47: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  r


=== Running: Fixed_Chunking_700_100 ===


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

In [4]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 2.2 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade langchain-groq langchain-core --no-cache-dir

In [ ]:
!pip install langchain-google-vertexai

In [ ]:
!pip uninstall -y langchain langchain-community langchain-openai
!pip install "langchain<1.0" "langchain-community<1.0" "langchain-openai<1.0" --no-cache-dir